# Broader-coverage skin condition classifier - Fitzpatrick17k

Second model for this project. The HAM10000 notebook (`01_...`) trained a high-precision
cancer-focused classifier on 7 lesion types. That's great for what it does but it has no
class for the everyday stuff people actually get - eczema, psoriasis, acne, keratosis pilaris,
rosacea. This notebook covers that.

I'm switching to the **Fitzpatrick17k** dataset (~17k images, 114 conditions). Two reasons:
1. It actually contains the common conditions HAM10000 ignores.
2. **Every image has a Fitzpatrick skin-type label (I-VI)**, which means I can finally
   *measure* whether the model is fair across skin tones - not just hope it is.

**Honest expectations before starting:**
- Top-1 accuracy will be much lower than the HAM10000 model. Going from 7 well-curated
  classes to ~30 messy ones is a real downgrade in top-1 even though it covers more cases.
  Top-3 is the realistic UX metric here (the app already shows top-3).
- Label quality is weaker than HAM10000 (atlas photos confirmed by a dermatologist viewing
  the case, not biopsy-confirmed).
- Many of the 114 conditions have very few images; I'm dropping those.
- Data download is uglier and slower than HAM10000 (per-image URL fetch, many URLs dead).

Plan:
- pull the CSV from the official repo, download images by URL (resumable, skips dead ones)
- drop classes with too few images (keep the trainable ones)
- stratified train/val/test split
- fine-tune EfficientNet-B0 with class-weighted loss
- evaluate overall macro-F1, top-3 accuracy
- **then the part that matters**: per-Fitzpatrick-type evaluation, to expose any skin-tone bias

Like the first notebook, this is still *not* a medical device. See `docs/ETHICS.md`.

## 0. Setup

In [ ]:
import sys, subprocess
def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
pip_install("seaborn", "scikit-learn", "requests")

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("No GPU detected. In Colab: Runtime -> Change runtime type -> T4 GPU.")

In [ ]:
import os, io, hashlib, random, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, UnidentifiedImageError

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import EfficientNet_B0_Weights

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, f1_score, top_k_accuracy_score
from sklearn.utils.class_weight import compute_class_weight

# keep things reproducible
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# the usual knobs
IMG_SIZE      = 224
BATCH_SIZE    = 32
EPOCHS        = 20          # this dataset benefits more from longer training than HAM10000
LR            = 1e-4
NUM_WORKERS   = 2
MIN_PER_CLASS = 100         # drop conditions with fewer than this many images (keeps things trainable)
DATA_DIR      = Path("data/fitzpatrick17k")
IMG_DIR       = DATA_DIR / "images"
CSV_URL       = "https://raw.githubusercontent.com/mattgroh/fitzpatrick17k/main/fitzpatrick17k.csv"
CSV_PATH      = DATA_DIR / "fitzpatrick17k.csv"
BEST_MODEL_PATH = Path("skin_classifier_fitz17k_efficientnet_b0.pt")

IMG_DIR.mkdir(parents=True, exist_ok=True)

## 1. Get the data

This is the painful part. Fitzpatrick17k isn't distributed as a single zip - the official
release is a CSV of URLs, and you fetch each image yourself. A bunch of those URLs are dead
by now (link rot from 2021); I just skip them and move on with whatever I can get.

The download is **resumable** - if Colab disconnects mid-fetch, re-running this cell picks
up where it left off (it checks which files already exist on disk).

Expect: ~10-25 minutes the first time, depending on how many URLs are alive. You'll typically
end up with 60-80% of the original 17k images.

In [ ]:
# Pull the metadata CSV.
if not CSV_PATH.exists():
    r = requests.get(CSV_URL, timeout=60)
    r.raise_for_status()
    CSV_PATH.write_bytes(r.content)
df = pd.read_csv(CSV_PATH)
print(f"CSV rows: {len(df)}")
print("columns:", list(df.columns))
df.head()

In [ ]:
def fname_for(row):
    # store every image as <md5hash>.jpg so re-runs can tell what's already downloaded
    return IMG_DIR / f"{row['md5hash']}.jpg"

def download_one(row):
    out = fname_for(row)
    if out.exists() and out.stat().st_size > 1024:   # already have it
        return "skip", out
    try:
        r = requests.get(row["url"], timeout=15, headers={"User-Agent": "skin-classifier-learning-project/0.1"})
        if r.status_code != 200 or len(r.content) < 1024:
            return "fail", out
        # verify it's actually a readable image (some URLs return HTML 200s)
        try:
            Image.open(io.BytesIO(r.content)).verify()
        except (UnidentifiedImageError, Exception):
            return "fail", out
        out.write_bytes(r.content)
        return "ok", out
    except Exception:
        return "fail", out

# Parallel download (URLs are the bottleneck, not local I/O). 16 workers is polite.
from tqdm.auto import tqdm
results = {"ok": 0, "skip": 0, "fail": 0}
rows = df.to_dict("records")
with ThreadPoolExecutor(max_workers=16) as pool:
    futures = [pool.submit(download_one, r) for r in rows]
    for fut in tqdm(as_completed(futures), total=len(futures), desc="downloading"):
        status, _ = fut.result()
        results[status] += 1

print(results)
print(f"images on disk: {len(list(IMG_DIR.glob('*.jpg')))}")

### Attach file paths + drop missing images + skip unknown skin types

In [ ]:
df["path"] = df["md5hash"].map(lambda h: IMG_DIR / f"{h}.jpg")
df["have"] = df["path"].map(lambda p: p.exists() and p.stat().st_size > 1024)
print(f"have image: {df['have'].sum()} / {len(df)}  ({df['have'].mean():.1%})")

df = df[df["have"]].reset_index(drop=True)

# Fitzpatrick scale: 1 (lightest) - 6 (darkest). -1 = unknown. Drop unknowns
# because we can't do fairness analysis on them.
df = df[df["fitzpatrick_scale"].between(1, 6)].reset_index(drop=True)
print(f"after dropping unknown skin types: {len(df)}")

# group into Light (I-II) / Medium (III-IV) / Dark (V-VI) for the fairness analysis
def fitz_group(s):
    s = int(s)
    if s <= 2: return "Light (I-II)"
    if s <= 4: return "Medium (III-IV)"
    return "Dark (V-VI)"
df["fitz_group"] = df["fitzpatrick_scale"].map(fitz_group)
print("\nskin tone distribution:")
print(df["fitz_group"].value_counts())

## 2. Look at the data

Two things I care about: which conditions have enough examples to be trainable, and how
the dataset is distributed across skin tones (the second is the whole point of using this
dataset).

In [ ]:
counts = df["label"].value_counts()
print(f"total conditions in CSV: {len(counts)}")
print(f"conditions with at least {MIN_PER_CLASS} images: {(counts >= MIN_PER_CLASS).sum()}")

plt.figure(figsize=(10, 4))
plt.hist(counts.values, bins=40, color="steelblue", edgecolor="white")
plt.axvline(MIN_PER_CLASS, color="red", linestyle="--", label=f"keep threshold ({MIN_PER_CLASS})")
plt.xlabel("images per condition"); plt.ylabel("# conditions"); plt.legend()
plt.title("Most conditions have very few images - this is why I drop the tail")
plt.tight_layout(); plt.show()

In [ ]:
# Cross-tab: which conditions also have decent representation across skin tones?
crosstab = pd.crosstab(df["label"], df["fitz_group"])
print("rows with >=100 images, sorted by total:")
print(crosstab[crosstab.sum(axis=1) >= MIN_PER_CLASS].sort_values(by=list(crosstab.columns), ascending=False).head(20))

## 3. Filter classes + build label index

In [ ]:
keep_labels = counts[counts >= MIN_PER_CLASS].index.tolist()
df = df[df["label"].isin(keep_labels)].reset_index(drop=True)
print(f"images after class filtering: {len(df)}")
print(f"classes after filtering: {df['label'].nunique()}")

CLASSES = sorted(df["label"].unique())
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
df["label_idx"] = df["label"].map(CLASS_TO_IDX)

print("\nfinal class list:")
for i, c in enumerate(CLASSES):
    print(f"  {i:2d}  {c}  ({(df['label'] == c).sum()} imgs)")

## 4. Stratified train/val/test split

Unlike HAM10000, Fitzpatrick17k doesn't have a `lesion_id` to group by, so I use a
stratified split on the class label (so every class shows up in every split).

In [ ]:
def strat_split(frame, target, test_frac, seed=SEED):
    sss = StratifiedShuffleSplit(n_splits=1, test_size=test_frac, random_state=seed)
    a_idx, b_idx = next(sss.split(frame, target))
    return frame.iloc[a_idx].reset_index(drop=True), frame.iloc[b_idx].reset_index(drop=True)

# 70/15/15
train_df, temp_df = strat_split(df, df["label_idx"], test_frac=0.30)
val_df, test_df   = strat_split(temp_df, temp_df["label_idx"], test_frac=0.50)
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

# Sanity check: skin tone proportions roughly preserved across splits
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    pct = d["fitz_group"].value_counts(normalize=True).round(2)
    print(f"  {name}: {pct.to_dict()}")

## 5. Datasets + augmentation + class weights

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class SkinDataset(Dataset):
    def __init__(self, frame, tf):
        self.frame = frame.reset_index(drop=True)
        self.tf = tf
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        try:
            img = Image.open(row["path"]).convert("RGB")
        except Exception:
            # a tiny number of files might be corrupt - return a black image rather than crash
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), color=(0, 0, 0))
        return self.tf(img), int(row["label_idx"])

train_ds = SkinDataset(train_df, train_tf)
val_ds   = SkinDataset(val_df, eval_tf)
test_ds  = SkinDataset(test_df, eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

class_weights = compute_class_weight("balanced", classes=np.arange(len(CLASSES)), y=train_df["label_idx"].values)
class_weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
print(f"class weights range: {class_weights.min().item():.2f} - {class_weights.max().item():.2f}")

## 6. Model - EfficientNet-B0, transfer learning again

In [ ]:
def build_model(num_classes):
    model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model

model = build_model(len(CLASSES)).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"EfficientNet-B0 ready: {len(CLASSES)} output classes, {n_params/1e6:.1f}M trainable params")

## 7. Training loop

Best model is picked by **validation macro-F1**, same logic as the first notebook.

In [ ]:
from tqdm.auto import tqdm

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
use_amp = (DEVICE.type == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

def run_epoch(loader, train):
    model.train() if train else model.eval()
    total_loss, all_preds, all_true = 0.0, [], []
    with torch.set_grad_enabled(train):
        for imgs, labels in tqdm(loader, leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if train:
                optimizer.zero_grad()
            with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
                out = model(imgs)
                loss = criterion(out, labels)
            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * imgs.size(0)
            all_preds.extend(out.argmax(1).detach().cpu().numpy())
            all_true.extend(labels.detach().cpu().numpy())
    avg_loss = total_loss / len(loader.dataset)
    macro_f1 = f1_score(all_true, all_preds, average="macro", zero_division=0)
    return avg_loss, macro_f1

In [ ]:
best_f1 = 0.0
history = {"train_loss": [], "val_loss": [], "val_f1": []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_f1 = run_epoch(train_loader, train=True)
    va_loss, va_f1 = run_epoch(val_loader, train=False)
    scheduler.step()
    history["train_loss"].append(tr_loss)
    history["val_loss"].append(va_loss)
    history["val_f1"].append(va_f1)
    flag = ""
    if va_f1 > best_f1:
        best_f1 = va_f1
        torch.save({"model_state": model.state_dict(),
                    "classes": CLASSES,
                    "img_size": IMG_SIZE}, BEST_MODEL_PATH)
        flag = "  <- saved best"
    print(f"epoch {epoch:2d}/{EPOCHS} | train_loss {tr_loss:.3f} | val_loss {va_loss:.3f} | val_macroF1 {va_f1:.3f}{flag}")

print(f"\nBest val macro-F1: {best_f1:.3f}  (saved to {BEST_MODEL_PATH})")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history["train_loss"], label="train"); ax[0].plot(history["val_loss"], label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(history["val_f1"], color="green"); ax[1].set_title("Validation macro-F1"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

## 8. Overall evaluation

Top-1 will look modest. Look at **top-3** - that's what the app actually shows the user.

In [ ]:
ckpt = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()

all_logits, all_true = [], []
with torch.no_grad():
    for imgs, labels in tqdm(test_loader, leave=False):
        logits = model(imgs.to(DEVICE)).cpu().numpy()
        all_logits.append(logits)
        all_true.extend(labels.numpy())

all_logits = np.concatenate(all_logits, axis=0)
all_probs  = torch.softmax(torch.tensor(all_logits), dim=1).numpy()
all_preds  = all_probs.argmax(axis=1)
all_true   = np.array(all_true)

top1 = (all_preds == all_true).mean()
top3 = top_k_accuracy_score(all_true, all_probs, k=3, labels=list(range(len(CLASSES))))
top5 = top_k_accuracy_score(all_true, all_probs, k=5, labels=list(range(len(CLASSES))))
macro_f1 = f1_score(all_true, all_preds, average="macro", zero_division=0)

print(f"top-1 accuracy:   {top1:.1%}")
print(f"top-3 accuracy:   {top3:.1%}   <- this is the UX metric, the app shows top-3")
print(f"top-5 accuracy:   {top5:.1%}")
print(f"test macro-F1:    {macro_f1:.3f}")

In [ ]:
print(classification_report(all_true, all_preds,
                            labels=list(range(len(CLASSES))),
                            target_names=CLASSES,
                            digits=3, zero_division=0))

## 9. Fairness analysis - performance across Fitzpatrick skin types

**This is the section that matters most.** Public dermatology datasets skew toward lighter
skin, and models trained on them are well-documented to underperform on darker skin. I'm
going to actually *measure* that here instead of just hoping it's fine.

For each Fitzpatrick group (Light / Medium / Dark) I look at:
- top-1 and top-3 accuracy
- macro-F1
- per-class recall, sorted to expose disparities

If accuracy drops meaningfully on darker skin, **the model is not safe to deploy and I need
to know that.** Being able to articulate this is more impressive than a high overall number.

In [ ]:
# Reattach fitz_group to test rows in the same order the loader iterated
test_fitz = test_df["fitz_group"].values

groups = ["Light (I-II)", "Medium (III-IV)", "Dark (V-VI)"]
rows = []
for g in groups:
    mask = (test_fitz == g)
    if mask.sum() == 0:
        rows.append({"group": g, "n": 0, "top1": np.nan, "top3": np.nan, "macroF1": np.nan})
        continue
    g_true   = all_true[mask]
    g_preds  = all_preds[mask]
    g_probs  = all_probs[mask]
    g_top1   = (g_preds == g_true).mean()
    g_top3   = top_k_accuracy_score(g_true, g_probs, k=3, labels=list(range(len(CLASSES))))
    g_f1     = f1_score(g_true, g_preds, average="macro", zero_division=0)
    rows.append({"group": g, "n": int(mask.sum()), "top1": g_top1, "top3": g_top3, "macroF1": g_f1})

fairness = pd.DataFrame(rows)
print(fairness.to_string(index=False))

In [ ]:
# Visual: side-by-side bars per metric, per group. Easy to read at a glance.
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, metric, title in zip(axes, ["top1", "top3", "macroF1"], ["Top-1", "Top-3", "Macro-F1"]):
    vals = fairness[metric].values
    bars = ax.bar(fairness["group"], vals, color=["#f4c95d", "#c47e3a", "#5e3a1c"])
    ax.set_title(title); ax.set_ylim(0, 1)
    for b, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(b.get_x() + b.get_width()/2, v + 0.01, f"{v:.1%}", ha="center", fontsize=9)
plt.suptitle("Performance across Fitzpatrick skin tones (test set)")
plt.tight_layout(); plt.show()

# Disparity summary
light = fairness.loc[fairness["group"] == "Light (I-II)"].iloc[0]
dark  = fairness.loc[fairness["group"] == "Dark (V-VI)"].iloc[0]
if not np.isnan(light["top3"]) and not np.isnan(dark["top3"]):
    gap = (light["top3"] - dark["top3"]) * 100
    print(f"top-3 accuracy gap (Light - Dark): {gap:+.1f} percentage points")
    if abs(gap) > 5:
        print("WARNING: meaningful disparity. The model is not equally reliable across skin tones.")

## 10. Single-image inference demo

In [ ]:
def predict_top_k(image, k=3):
    if not isinstance(image, Image.Image):
        image = Image.open(image).convert("RGB")
    x = eval_tf(image).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1).squeeze(0).cpu().numpy()
    top = probs.argsort()[::-1][:k]
    return [(CLASSES[i], float(probs[i])) for i in top]

# pick a random test image
sample = test_df.sample(1, random_state=SEED).iloc[0]
preds = predict_top_k(sample["path"], k=3)

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].imshow(Image.open(sample["path"]).convert("RGB")); ax[0].axis("off")
ax[0].set_title(f"true: {sample['label']}\nskin type: {sample['fitz_group']}", fontsize=9)
labels = [c for c, _ in preds][::-1]
vals = [p for _, p in preds][::-1]
ax[1].barh(labels, vals, color="steelblue"); ax[1].set_xlim(0, 1)
ax[1].set_title("Top-3 guesses"); ax[1].set_xlabel("probability")
for i, v in enumerate(vals):
    ax[1].text(v + 0.01, i, f"{v:.0%}", va="center", fontsize=9)
plt.tight_layout(); plt.show()

## 11. Save the model

In [ ]:
print("saved best model to:", BEST_MODEL_PATH.resolve(),
      f"({BEST_MODEL_PATH.stat().st_size/1e6:.1f} MB)")
# To download from Colab:
#   from google.colab import files; files.download(str(BEST_MODEL_PATH))

## 12. Limitations + next steps

What this model adds vs. the HAM10000 one:
- Covers ~30 common skin conditions instead of 7 cancer-focused ones (so an everyday photo
  of acne or pilaris stands a chance of being classified usefully).
- Comes with a real fairness measurement across Fitzpatrick skin tones.

What it gives up:
- Top-1 accuracy is much lower than the HAM10000 model. Use top-3 as the headline UX metric.
- Labels are weaker (atlas-confirmed, not biopsy-confirmed).
- Some classes still don't have enough images for the model to learn them well.

In the desktop app I'm going to keep both models loadable:
- **HAM10000 model** -> a focused "cancer triage" specialist.
- **Fitzpatrick17k model** -> a broader everyday classifier with a fairness report attached.

The honest framing: *"here's a high-precision narrow model and a broad-coverage messy model,
here's exactly when each one fails, and here's how I measured the fairness of the broad one."*

Next experiments worth trying:
1. Grad-CAM on both models, side by side.
2. **Calibrated** confidence (so a "60%" actually means 60%). Probably temperature scaling.
3. An out-of-distribution detector that says "this doesn't look like skin" instead of
   guessing on a photo of a wall.